# Spatial-Temporal Dynamics of Pakistan's Air Quality (2015–2030)

**Author:** Mohammad Rabbani · **Domain:** Environmental analytics and public-health risk

## Abstract
An 11-year city panel (2015–2025; 10 cities; 110 city-years) is used to quantify three things: the spatial concentration of PM2.5 in the Punjab smog belt, the winter-inversion multiplier, and exceedance of the 2021 WHO annual guideline (5 µg/m³). Linear trends are then estimated for every city **with 95% confidence and prediction intervals**, and checked with a non-parametric Kendall test, before any projection to 2030 is shown.

**Data:** annual city-level summary compiled by the author ([dataset](https://www.kaggle.com/datasets/rabbani0123/pakistan-air-quality-annual-summary-2015-2025), [GitHub](https://github.com/Rabbani-bot/Pakistan-air-quality-modeling)). Values are annual summaries, so they should be cross-checked against primary monitoring records before being used for regulatory decisions.

In [ ]:
import glob, os, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#E6E8EC", "axes.titleweight": "bold", "axes.titlelocation": "left"})
WHO = 5.0  # µg/m³, WHO 2021 annual guideline

hits = glob.glob("/kaggle/input/**/pakistan_air_quality_annual_summary.csv", recursive=True)
if hits:
    path = hits[0]
else:
    if not os.path.exists("Pakistan-air-quality-modeling"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Rabbani-bot/Pakistan-air-quality-modeling.git"], check=True)
    path = "Pakistan-air-quality-modeling/pakistan_air_quality_annual_summary.csv"
df = pd.read_csv(path)
print(f"{len(df)} city-years | {df.city.nunique()} cities | {df.year.min()}–{df.year.max()}")
df.head()

## 0. Data checks

In [ ]:
checks = {
    "complete panel (city × year)": df.groupby("city").year.nunique().eq(df.year.nunique()).all(),
    "no missing PM2.5": df.pm25_annual_avg.notna().all(),
    "min ≤ annual mean ≤ peak": ((df.pm25_min_month <= df.pm25_annual_avg) & (df.pm25_annual_avg <= df.pm25_peak_month)).all(),
    "who_ratio = mean / 5 (±0.1)": np.allclose(df.who_ratio_annual, df.pm25_annual_avg / WHO, atol=0.1),
}
pd.Series(checks, name="passed").to_frame()

## 1. Spatial disparity: the Punjab smog belt (2025)

In [ ]:
d25 = df[df.year == 2025].sort_values("pm25_annual_avg", ascending=False)
fig, ax = plt.subplots(figsize=(11, 4.8))
cols = ["#C0392B" if p == "Punjab" else "#34495E" for p in d25.province]
bars = ax.bar(d25.city, d25.pm25_annual_avg, color=cols)
ax.axhline(WHO, color="#2980B9", ls="--", lw=1.5, label="WHO 2021 annual guideline (5 µg/m³)")
for b, r in zip(bars, d25.who_ratio_annual):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 2, f"{b.get_height():.0f}\n({r:.0f}×)", ha="center", fontsize=8.5)
ax.set_ylabel("Annual mean PM2.5 (µg/m³)"); ax.set_title("Annual mean PM2.5 by city, 2025 (red = Punjab)")
ax.legend(frameon=False); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

punjab = d25[d25.province == "Punjab"].pm25_annual_avg
other = d25[d25.province != "Punjab"].pm25_annual_avg
u, p = stats.mannwhitneyu(punjab, other, alternative="greater")
print(f"Punjab median {punjab.median():.1f} vs other provinces {other.median():.1f} µg/m³ "
      f"(Mann–Whitney U = {u:.0f}, one-sided p = {p:.3f}; n = {len(punjab)} vs {len(other)})")
print(f"Every city exceeds the WHO guideline; range {d25.who_ratio_annual.min():.0f}× to {d25.who_ratio_annual.max():.0f}×.")

## 2. The winter-inversion multiplier (2025)

In [ ]:
s = df[df.year == 2025].assign(multiplier=lambda x: x.pm25_peak_month / x.pm25_min_month).sort_values("pm25_peak_month")
fig, ax = plt.subplots(figsize=(11, 5))
y = np.arange(len(s))
ax.hlines(y, s.pm25_min_month, s.pm25_peak_month, color="#BDC3C7", lw=3)
ax.scatter(s.pm25_min_month, y, color="#2ECC71", s=70, zorder=3, label="Cleanest month")
ax.scatter(s.pm25_peak_month, y, color="#E74C3C", s=70, zorder=3, label="Peak (winter) month")
for yi, (pk, m) in enumerate(zip(s.pm25_peak_month, s.multiplier)):
    ax.text(pk + 4, yi, f"{m:.1f}×", va="center", fontsize=9)
ax.set_yticks(y); ax.set_yticklabels(s.city); ax.set_xlabel("Monthly mean PM2.5 (µg/m³)")
ax.set_title("Seasonal range and peak-to-minimum multiplier, 2025"); ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.show()
print(f"Median winter multiplier: {s.multiplier.median():.1f}× (range {s.multiplier.min():.1f}–{s.multiplier.max():.1f}×)")

## 3. Trends 2015–2025 with uncertainty
For each city, PM2.5 is regressed on year by ordinary least squares (OLS), giving a slope in µg/m³ per year with a 95% CI and *p*-value. Kendall's τ is a rank-based check that does not assume linearity or normal errors. With *n* = 11 annual points, wide intervals are expected.

In [ ]:
def ols(x, y):
    n = len(x); X = np.column_stack([np.ones(n), x - x.mean()])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta; s2 = resid @ resid / (n - 2)
    cov = s2 * np.linalg.inv(X.T @ X)
    return beta, cov, s2, x.mean(), n

rows = []
for city, g in df.sort_values("year").groupby("city"):
    x, yv = g.year.to_numpy(float), g.pm25_annual_avg.to_numpy(float)
    beta, cov, s2, xm, n = ols(x, yv)
    se = np.sqrt(cov[1, 1]); t = stats.t.ppf(0.975, n - 2)
    tau, pk = stats.kendalltau(x, yv)
    rows.append(dict(city=city, slope=beta[1], ci_low=beta[1] - t * se, ci_high=beta[1] + t * se,
                     p_value=2 * stats.t.sf(abs(beta[1] / se), n - 2), kendall_tau=tau, kendall_p=pk,
                     r2=1 - s2 * (n - 2) / ((yv - yv.mean()) ** 2).sum()))
trend = pd.DataFrame(rows).set_index("city").sort_values("slope").round(3)
trend

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
yy = np.arange(len(trend))
sig = trend.p_value < 0.05
ax.errorbar(trend.slope, yy, xerr=[trend.slope - trend.ci_low, trend.ci_high - trend.slope], fmt="none", ecolor="#8A94A6", capsize=3)
ax.scatter(trend.slope[sig], yy[sig], color="#C0392B", zorder=3, label="p < 0.05")
ax.scatter(trend.slope[~sig], yy[~sig], facecolor="white", edgecolor="#34495E", zorder=3, label="not significant")
ax.axvline(0, color="#5C677D", ls="--", lw=1)
ax.set_yticks(yy); ax.set_yticklabels(trend.index); ax.set_xlabel("OLS trend, µg/m³ per year (95% CI)")
ax.set_title("Is PM2.5 changing? City trends 2015–2025"); ax.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()
print(f"{int(sig.sum())} of {len(trend)} cities show a statistically significant linear trend at the 5% level.")

## 4. Projection to 2030 with 95% prediction intervals
Projections extend the fitted line for four sentinel cities. The shaded band is the **95% prediction interval** for an individual future year, which widens with distance from the data. These are trend extrapolations, not forecasts of policy or meteorology.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
pal = {"Faisalabad": "#7D6608", "Lahore": "#78281F", "Karachi": "#1B4F72", "Islamabad": "#145A32"}
fut = np.arange(2015, 2031, dtype=float)
proj = []
for city, col in pal.items():
    g = df[df.city == city].sort_values("year")
    x, yv = g.year.to_numpy(float), g.pm25_annual_avg.to_numpy(float)
    beta, cov, s2, xm, n = ols(x, yv)
    Xf = np.column_stack([np.ones_like(fut), fut - xm]); yhat = Xf @ beta
    pse = np.sqrt(s2 + np.einsum("ij,jk,ik->i", Xf, cov, Xf)); t = stats.t.ppf(0.975, n - 2)
    m = fut > 2025
    ax.plot(x, yv, "o-", color=col, lw=1.8, label=city)
    ax.plot(fut[m], yhat[m], ":", color=col, lw=2)
    ax.fill_between(fut[m], yhat[m] - t * pse[m], yhat[m] + t * pse[m], color=col, alpha=0.12)
    proj.append(dict(city=city, pm25_2030=yhat[-1], pi_low=yhat[-1] - t * pse[-1], pi_high=yhat[-1] + t * pse[-1]))
ax.axhline(WHO, color="#C0392B", ls="--", lw=1.2, label="WHO guideline (5 µg/m³)")
ax.set_xlim(2015, 2030); ax.set_ylabel("Annual mean PM2.5 (µg/m³)"); ax.set_xlabel("Year")
ax.set_title("Observed 2015–2025 and linear projection to 2030 (shaded: 95% prediction interval)")
ax.legend(frameon=False, ncol=5, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.12)); plt.tight_layout(); plt.show()
pd.DataFrame(proj).set_index("city").round(1)

## 5. Conclusions
- **Exceedance is universal and large.** Every city exceeds the WHO annual guideline by a wide multiple in 2025, highest in the Punjab smog belt.
- **Winter is the critical season.** Peak-month concentrations are several times the cleanest month, so annual means understate acute exposure.
- **Trends are uncertain.** Few city trends are statistically distinguishable from zero over 11 years, and 2030 prediction intervals are wide. No plausible trajectory reaches the WHO guideline by 2030 without structural intervention.

## 6. Limitations
- Annual summaries hide within-year variability and monitor-network changes.
- Linear trends cannot capture policy shocks, meteorology or COVID-19-period effects (2020–2021).
- City values are point estimates without measurement uncertainty; results should be validated against primary monitoring data.